# pywhyllm — Sea Ice Causality Example

A realistic causal discovery workflow over Arctic sea ice climate variables.

**What this notebook covers:**
- Building a causal graph over 10 climate variables
- Expert consensus (parallel calls)
- Local graph queries — zero extra LLM calls
- Confounders, latent confounders, negative controls
- Graph validation via critique
- API call count vs the old pairwise approach

In [ ]:
%pip install -e ../.. instructor openai python-dotenv --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # reads OPENAI_API_KEY from a .env file

# Or set directly for quick testing (don't commit real keys!):
import os; os.environ["OPENAI_API_KEY"] = ""

In [ ]:
import instructor
from openai import AsyncOpenAI

from pywhyllm.suggesters import ModelSuggester, CausalGraph

client = instructor.from_openai(AsyncOpenAI())

suggester = ModelSuggester(
    client=client,
    model="gpt-5.4-mini-2026-03-17",
    context="Arctic climate systems and sea ice dynamics",
)

## Variables

Ten atmospheric and oceanic variables relevant to Arctic sea ice extent.

In [ ]:
variables = [
    "geopotential_heights",
    "relative_humidity",
    "sea_level_pressure",
    "zonal_wind_at_10_meters",
    "meridional_wind_at_10_meters",
    "total_precipitation",
    "surface_temperature",
    "sea_ice_extent",
    "ocean_heat_content",
    "longwave_radiation",
]

# Treatment and outcome of interest
treatment = "surface_temperature"
outcome   = "sea_ice_extent"

## Step 1 — Suggest domain experts

One LLM call returns expert roles relevant to causal reasoning over these variables.

In [ ]:
experts = await suggester.suggest_domain_experts(variables, n_experts=3)
print("Experts:", experts)

## Step 2 — Build the causal graph

**Without experts**: 1 LLM call, generic persona.  
**With experts**: 1 call *per expert*, fired in parallel — edges merged by vote count.

For 10 variables the old pairwise approach would have made **10C2 × 3 = 135 sequential calls**.  
The new approach makes **3 parallel calls** — the variable count is irrelevant.

In [ ]:
# 3 parallel calls — one per expert
graph = await suggester.suggest_graph(variables, expertise_list=experts)
print(graph)

## Step 3 — Filter by consensus

`top_edges(min_votes=2)` keeps only edges at least 2 experts agreed on. Zero LLM calls.

In [ ]:
consensus_edges = graph.top_edges(min_votes=2)
print(f"{len(consensus_edges)} edges with ≥2 expert votes:\n")
for (cause, effect), data in consensus_edges:
    print(f"  {cause} → {effect}  (votes: {data.votes}, avg confidence: {data.avg_confidence:.2f})")

## Step 4 — Local graph queries

All structural queries are local — **zero LLM calls** after the graph is built.

In [ ]:
print("Parents of sea_ice_extent:",   graph.parents_of("sea_ice_extent"))
print("Children of surface_temperature:", graph.children_of("surface_temperature"))
print("Ancestors of sea_ice_extent:",  graph.ancestors_of("sea_ice_extent"))
print()
print("Mediators (surface_temperature → sea_ice_extent):",
      graph.mediators_of(treatment, outcome))
print("Instrumental variables (surface_temperature → sea_ice_extent):",
      graph.instrumental_variables_for(treatment, outcome))

## Step 5 — Confounders

Find variables that directly cause **both** treatment and outcome.

`latent=True` asks the LLM for unmeasured confounders *outside* the variable list.

In [ ]:
confounders = await suggester.suggest_confounders(
    treatment=treatment,
    outcome=outcome,
    variables=variables,
    expertise_list=experts,   # 3 parallel calls, results deduplicated
)
print("Observed confounders:", confounders)

In [ ]:
latent = await suggester.suggest_confounders(
    treatment=treatment,
    outcome=outcome,
    variables=variables,
    latent=True,
)
print("Latent confounders:", latent)

## Step 6 — Negative controls

Variables that *should* be unaffected by `surface_temperature`. If your model shows a treatment effect on these, your causal assumptions are wrong.

In [ ]:
negative_controls = await suggester.suggest_negative_controls(
    treatment=treatment,
    outcome=outcome,
    variables=variables,
)
print("Negative controls:", negative_controls)

## Step 7 — Critique the graph

Show the LLM the proposed edges and ask: *which are genuinely causal, and what's missing?*  
Returns a new graph — compare against the original to find confirmed vs disputed edges.

In [ ]:
critique = await suggester.critique_graph(
    graph=graph,
    variables=variables,
    expertise_list=experts,   # 3 parallel calls
)

original_edges = set(graph.edges)
for (cause, effect) in sorted(original_edges):
    status = "✓ CONFIRMED" if critique.edge_data(cause, effect) else "✗ DISPUTED"
    print(f"  {status}: {cause} → {effect}")

new_edges = [(c, e) for (c, e) in critique.edges if (c, e) not in original_edges]
if new_edges:
    print("\nNew edges suggested by critique:")
    for (cause, effect) in new_edges:
        print(f"  + {cause} → {effect}")

## Step 8 — Inspect edge reasoning

Every edge stores per-expert reasoning. Useful for auditing *why* the model believes a relationship exists.

In [ ]:
data = graph.edge_data("surface_temperature", "sea_ice_extent")
if data:
    print(f"Votes: {data.votes}")
    print(f"Avg confidence: {data.avg_confidence:.2f}")
    print("Expert reasoning:")
    for i, r in enumerate(data.reasonings, 1):
        print(f"  [{i}] {r}")
else:
    print("Edge not in graph.")

## Step 9 — Visualise the graph

`plot()` uses Graphviz's `dot` engine, which topologically sorts nodes into ranks so causal direction reads left-to-right naturally.

- **Edge thickness + colour** → average confidence (dark blue = high, grey = lower)
- **Edge label** → confidence %; vote count shown when using multiple experts
- **Node labels** → underscores replaced with spaces for readability

Requires: `pip install graphviz` + `brew install graphviz` (macOS) / `apt-get install graphviz` (Linux)

In [ ]:
# Render inline in Jupyter
graph.plot()

In [ ]:
# Consensus edges only (≥2 expert votes), top-to-bottom layout
graph.plot(min_votes=2, rankdir="TB")

In [ ]:
# Save to file
graph.plot(filename="sea_ice_causal_graph", format="png")